# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [2]:
aviation_df = pd.read_csv("data/AviationData.csv", encoding="latin-1")

aviation_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Number         88889 non-null  object 
 3   Event.Date              88889 non-null  object 
 4   Location                88837 non-null  object 
 5   Country                 88663 non-null  object 
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  object 
 9   Airport.Name            52704 non-null  object 
 10  Injury.Severity         87889 non-null  object 
 11  Aircraft.damage         85695 non-null  object 
 12  Aircraft.Category       32287 non-null  object 
 13  Registration.Number     87507 non-null  object 
 14  Make                    88826 non-null

/var/folders/y_/8lgqgn_17j53v04m52jxzw7c0000gn/T/ipykernel_58147/364085468.py:1: DtypeWarning: Columns (6,7,28) have mixed types. Specify dtype option on import or set low_memory=False.
  aviation_df = pd.read_csv("data/AviationData.csv", encoding="latin-1")


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [3]:
aviation_df = aviation_df[aviation_df["Aircraft.Category"] == "Airplane"]
aviation_df = aviation_df[aviation_df["Amateur.Built"] == "No"]
aviation_df = aviation_df[pd.to_datetime(aviation_df["Event.Date"]).dt.year >= 1983]

#aviation_df.info()
aviation_df.loc[:, ["Aircraft.Category", "Amateur.Built", "Event.Date"]]

,Aircraft.Category,Amateur.Built,Event.Date
4149,Airplane,No,1983-03-18
4150,Airplane,No,1983-03-18
4171,Airplane,No,1983-03-20
4285,Airplane,No,1983-04-02
5957,Airplane,No,1983-08-21
...,...,...,...
88869,Airplane,No,2022-12-13
88873,Airplane,No,2022-12-14
88876,Airplane,No,2022-12-15
88877,Airplane,No,2022-12-16


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [4]:
#Injury.Severity records either fatal or non-fatal, but does not count serious injuries, so the Total.Fatal.Injuries and Total.Serious.Injuries must be added manually.

# Clean passenger condition data, then sum passenger conditions accross columns and store to "Passengers" column.
aviation_df.loc[:, "Total.Fatal.Injuries":"Total.Uninjured"] = aviation_df.loc[:, "Total.Fatal.Injuries":"Total.Uninjured"].fillna(0)
aviation_df["Number.of.Passengers"] = aviation_df.loc[:, "Total.Fatal.Injuries":"Total.Uninjured"].sum(axis=1)
# Severity.Likelihood is a value [0:1]: sum of fatal and serious injuries divided by number of passengers.
aviation_df["Severity.Likelihood"] = aviation_df.loc[:, "Total.Fatal.Injuries":"Total.Serious.Injuries"].sum(axis=1) / aviation_df["Number.of.Passengers"]
aviation_df["Severity.Likelihood"] = aviation_df["Severity.Likelihood"].fillna(0)

aviation_df.loc[:, ["Total.Serious.Injuries", "Total.Fatal.Injuries", "Number.of.Passengers","Severity.Likelihood"]]

,Total.Serious.Injuries,Total.Fatal.Injuries,Number.of.Passengers,Severity.Likelihood
4149,0.0,0.0,588.0,0.0
4150,0.0,0.0,588.0,0.0
4171,1.0,1.0,2.0,1.0
4285,0.0,1.0,5.0,0.2
5957,0.0,0.0,289.0,0.0
...,...,...,...,...
88869,0.0,0.0,1.0,0.0
88873,0.0,0.0,1.0,0.0
88876,0.0,0.0,1.0,0.0
88877,1.0,0.0,1.0,1.0


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [5]:
aviation_df["Aircraft.damage"] = aviation_df["Aircraft.damage"].fillna("Unknown")
aviation_df["Aircraft.Destroyed"] = aviation_df["Aircraft.damage"] == "Destroyed"

aviation_df.loc[:, ["Aircraft.damage", "Aircraft.Destroyed"]]

,Aircraft.damage,Aircraft.Destroyed
4149,Minor,False
4150,Minor,False
4171,Destroyed,True
4285,Unknown,False
5957,Minor,False
...,...,...
88869,Substantial,False
88873,Substantial,False
88876,Substantial,False
88877,Substantial,False


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [6]:
## Make each Make in UPPERCASE
aviation_df["Make"] = aviation_df["Make"].str.upper()

# Remove periods and commas
aviation_df["Make"] = aviation_df["Make"].str.replace('.', '')
aviation_df["Make"] = aviation_df["Make"].str.replace(',', '')

# Remove the suffix of each Make, as these are inconsistent in the dataset,
# and can cause same Make to be considered different
# Use regex $ so INC, LTD, etc. is only removed if at the end
aviation_df["Make"] = aviation_df["Make"].str.replace(r'\s*(INC|LTD|LLC|CORP|CORPORATION|AVIATION|COMPANY|CO|INCORPORATE|INCORPORATED)\.?$', '', regex=True)

# Strip whitespace from beginning and end of Make
aviation_df["Make"] = aviation_df["Make"].str.strip()

# Keep the Makes that have atleast 50 entries
make_counts = aviation_df["Make"].value_counts()
aviation_df = aviation_df[aviation_df["Make"].map(make_counts) >= 50]

aviation_df.loc[:, "Make"]

4150                         BOEING
4171                          PIPER
4285                   DE HAVILLAND
6760                         BOEING
6806                          BEECH
                    ...            
88865                        CESSNA
88869                         PIPER
88873                 CIRRUS DESIGN
88877                        CESSNA
88886    AMERICAN CHAMPION AIRCRAFT
Name: Make, Length: 18130, dtype: object

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [7]:
# for i in aviation_df["Model"].sort_values().unique():
#     print(i)

aviation_df = aviation_df.dropna(subset='Model')

# print(aviation_df["Model"].value_counts().head())
# print(aviation_df["Make"].value_counts().head())

aviation_df["Aircraft.Type"] = aviation_df["Make"] + ' ' + aviation_df["Model"]

aviation_df.loc[:, ["Make", "Model", "Aircraft.Type"]]

#viation_df.loc[aviation_df["Model"] == aviation_df["Model"].mode()[0], ["Make", "Model"]]
#aviation_df

,Make,Model,Aircraft.Type
4150,BOEING,747,BOEING 747
4171,PIPER,PA-28-140,PIPER PA-28-140
4285,DE HAVILLAND,DHC-6,DE HAVILLAND DHC-6
6760,BOEING,727-200,BOEING 727-200
6806,BEECH,C35,BEECH C35
...,...,...,...
88865,CESSNA,172,CESSNA 172
88869,PIPER,PA42,PIPER PA42
88873,CIRRUS DESIGN,SR22,CIRRUS DESIGN SR22
88877,CESSNA,R172K,CESSNA R172K


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [8]:
aviation_df["Engine.Type"] = aviation_df["Engine.Type"].replace("UNK", "Unknown")
# aviation_df["Engine.Type"].unique()

aviation_df["Weather.Condition"] = aviation_df["Weather.Condition"].str.upper()
aviation_df["Weather.Condition"] = aviation_df["Weather.Condition"].replace("UNK", "Unknown")
# aviation_df["Weather.Condition"].unique()

# Number.of.Engines is clean
#aviation_df["Number.of.Engines"].unique()

aviation_df["Purpose.of.flight"] = aviation_df["Purpose.of.flight"].replace("Air Race show", "Air Race/show")
# aviation_df["Purpose.of.flight"].unique()

# Broad.phase.of.flight is clean
# aviation_df["Broad.phase.of.flight"].unique()

aviation_df.loc[:, ["Engine.Type", "Weather.Condition", "Number.of.Engines", "Purpose.of.flight", "Broad.phase.of.flight"]]

,Engine.Type,Weather.Condition,Number.of.Engines,Purpose.of.flight,Broad.phase.of.flight
4150,Turbo Fan,VMC,4.0,NaN,Taxi
4171,Reciprocating,IMC,1.0,Personal,Cruise
4285,Turbo Prop,VMC,2.0,Skydiving,Standing
6760,Turbo Fan,VMC,3.0,Unknown,Taxi
6806,Reciprocating,VMC,1.0,Personal,Climb
...,...,...,...,...,...
88865,NaN,VMC,1.0,Instructional,NaN
88869,NaN,NaN,2.0,NaN,NaN
88873,NaN,VMC,1.0,Personal,NaN
88877,NaN,VMC,1.0,Personal,NaN


### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [9]:
print(f"before: {aviation_df.columns}")

aviation_df = aviation_df.dropna(thresh=len(aviation_df) * 0.5, axis=1)

print(f"after: {aviation_df.columns}")

before: Index(['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date',
       'Location', 'Country', 'Latitude', 'Longitude', 'Airport.Code',
       'Airport.Name', 'Injury.Severity', 'Aircraft.damage',
       'Aircraft.Category', 'Registration.Number', 'Make', 'Model',
       'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'FAR.Description',
       'Schedule', 'Purpose.of.flight', 'Air.carrier', 'Total.Fatal.Injuries',
       'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured',
       'Weather.Condition', 'Broad.phase.of.flight', 'Report.Status',
       'Publication.Date', 'Number.of.Passengers', 'Severity.Likelihood',
       'Aircraft.Destroyed', 'Aircraft.Type'],
      dtype='object')
after: Index(['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date',
       'Location', 'Country', 'Latitude', 'Longitude', 'Airport.Code',
       'Airport.Name', 'Injury.Severity', 'Aircraft.damage',
       'Aircraft.Category', 'Registration.Number', 'Make',

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [ ]:
aviation_df.to_csv('Aviation_Cleaned.csv', index=False)

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Report.Status,Publication.Date,Number.of.Passengers,Severity.Likelihood,Aircraft.Destroyed,Aircraft.Type
4150,20001214X42478,Incident,LAX83IA149A,1983-03-18,"LOS ANGELES, CA",United States,NaN,NaN,LAX,LOS ANGELES INTL,...,0.0,0.0,588.0,VMC,Probable Cause,04-12-2014,588.0,0.0,False,BOEING 747
4171,20001214X42331,Accident,ATL83FA140,1983-03-20,"CROSSVILLE, TN",United States,NaN,NaN,NaN,NaN,...,1.0,0.0,0.0,IMC,Probable Cause,02-05-2011,2.0,1.0,True,PIPER PA-28-140
4285,20001214X42672,Accident,FTW83LA177,1983-04-02,"MCKINNEY, TX",United States,NaN,NaN,TX05,AERO COUNTRY,...,0.0,0.0,4.0,VMC,Probable Cause,17-10-2016,5.0,0.2,False,DE HAVILLAND DHC-6
6760,20001214X45013,Incident,CHI84IA041,1983-11-08,"CHICAGO, IL",United States,NaN,NaN,ORD,O'HARE,...,0.0,0.0,100.0,VMC,Probable Cause,11-06-2018,100.0,0.0,False,BOEING 727-200
6806,20001214X45188,Accident,NYC84LA028,1983-11-13,"MARTHA'S VINEYARD, MA",United States,NaN,NaN,NaN,NaN,...,0.0,0.0,1.0,VMC,Probable Cause,05-05-2011,1.0,0.0,False,BEECH C35
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88865,20221212106444,Accident,ERA23LA085,2022-12-12,"Knoxville, TN",United States,355745N,0835218W,DKX,KNOXVILLE DOWNTOWN ISLAND,...,0.0,0.0,1.0,VMC,NaN,15-12-2022,1.0,0.0,False,CESSNA 172
88869,20221213106455,Accident,WPR23LA065,2022-12-13,"Lewistown, MT",United States,047257N,0109280W,KLWT,Lewiston Municipal Airport,...,0.0,0.0,1.0,NaN,NaN,14-12-2022,1.0,0.0,False,PIPER PA42
88873,20221215106463,Accident,ERA23LA090,2022-12-14,"San Juan, PR",United States,182724N,0066554W,SIG,FERNANDO LUIS RIBAS DOMINICCI,...,0.0,0.0,1.0,VMC,NaN,27-12-2022,1.0,0.0,False,CIRRUS DESIGN SR22
88877,20221219106470,Accident,ERA23LA091,2022-12-16,"Brooksville, FL",United States,282825N,0822719W,BKV,BROOKSVILLE-TAMPA BAY RGNL,...,1.0,0.0,0.0,VMC,NaN,23-12-2022,1.0,1.0,False,CESSNA R172K
